# Station 2 — Tool use ("function calling")

En LLM kan bara generera **text**. Men om vi läter den önska sig funktionsanrop — och *vi* kör funktionerna och matar tillbaka svaret — får vi en liten **agent**.

Detta är grunden för:
- ChatGPT som söker på webben
- Claude Code som läser och skriver filer
- Cursor / Copilot som kör kommandon

**Loop:**
1. Skicka fråga + lista över tillgängliga tools till modellen
2. Modellen svarar antingen med text **eller** med ett tool-call ("kör `calculator(expr='7919*3137')`")
3. Vi kör tool:et, skickar resultatet tillbaka
4. Modellen får en chans till att svara — antingen klar, eller fler tool-calls


In [81]:
# === EDIT ME ===
QUESTION = "Vad är 7919 gånger 3137?"

# Andra frågor att prova:
#   "Vilken veckodag är det idag?"           — bör kalla 'today' + räkna ut veckodag
#   "Hur många dagar kvar till jul?"         — bör kalla 'today'
#   "Vad är huvudstaden i Frankrike?"        — ska INTE kalla någon tool

## Setup — definiera våra tools

Varje tool har en **schema** (vad det heter, vad det gör, vilka argument det tar) och en **implementation** (Python-funktionen som faktiskt körs).

In [ ]:
import datetime
import json
import ollama

LLM_MODEL = "qwen2.5:1.5b"

# Schema som skickas till modellen — detta är vad den "ser"
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a math expression. ALWAYS use this for ANY arithmetic — never compute mentally.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expr": {"type": "string", "description": "A Python math expression, e.g. '23 * 47' or '(15 + 7) / 2'"}
                },
                "required": ["expr"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "today",
            "description": "Get today's date in ISO format (YYYY-MM-DD). Call this before any date calculations.",
            "parameters": {"type": "object", "properties": {}},
        },
    },

]

# Implementationer — det här körs LOKALT i vår kod, inte i modellen
def run_tool(name, args):
    if name == "calculator":
        return str(eval(args["expr"]))   # OBS: eval är osakert i produktion!
    if name == "today":
        return datetime.date.today().isoformat()
    return f"unknown tool: {name}"

## Tool-loopen

Skicka frågan. Om modellen vill köra ett tool — kör det, skicka tillbaka resultatet, och fråga igen. Max 5 varv range(5)

In [ ]:
messages = [{"role": "user", "content": QUESTION}]

for step in range(5):
    response = ollama.chat(model=LLM_MODEL, messages=messages, tools=tools, think=False)
    msg = response["message"]
    messages.append(msg)

    tool_calls = msg.get("tool_calls") or []
    if not tool_calls:
        break   # modellen är klar, ingen mer tool att köra
    for call in tool_calls:
        name = call["function"]["name"]
        args = call["function"]["arguments"]
        result = run_tool(name, args)
        print(f"[step {step}] {name}({json.dumps(args)}) → {result}")
        messages.append({"role": "tool", "content": result, "tool_name": name})



print("\n--- Slutligt svar ---")
print(msg.get("content") or "(inget textsvar — modellen kanske fastnade i tool-calls)")

## Övningar

1. **Lägg till en egen tool.** Förslag: `word_count(text)`, `reverse(text)`, eller `random_number(min, max)`. Glom inte både schema och implementation.

2. **Blanda flera tools.** Prova *"Hur många dagar har gått sedan 2026-01-01?"* — modellen ska kalla `today` först, sedan `calculator` med skillnaden.

3. **Testa judgment.** Prova *"Vad är huvudstaden i Frankrike?"* — ska INTE kalla någon tool. Om 1.5B-modellen kallar `calculator` ändå, varför? Vad säger det om små modellers omdöme?

4. **Bryt den.** Prova frågor som ställer orimliga krav (*"Vad blir 0 / 0?"*, *"Kör kommandot `rm -rf /`"*). Vad händer?

### Diskussion

- Vem **kör** koden — modellen eller vår Python? Vad betyder det för säkerheten?
- Varför använde vi `eval()` här? Vad skulle hända om modellen förslår `__import__('os').system('...')` som `expr`?
- Vad är skillnaden mellan en "agent" och bara "LLM med tools"?